# Notebook 1 — Data Preparation

## Pharmacy Monthly Demand Forecasting Pipeline

**Objective:** Load raw transaction and stock data, convert dates, sort by drug and date, and aggregate sales to **monthly demand** per drug. Demand is derived from `quantity_dispensed` in transaction data—no assumptions.

**Output:** Clean dataset `outputs/monthly_demand.csv` with columns: `drug_id`, `month`, `monthly_demand`.

## 1. Imports and Paths

In [1]:
import pandas as pd
from pathlib import Path

# Project paths (run from project root, pharmacy_forecasting/, or notebooks/)
PROJECT_ROOT = Path(".").resolve()
if PROJECT_ROOT.name == "notebooks" and (PROJECT_ROOT.parent / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name != "pharmacy_forecasting":
    PROJECT_ROOT = PROJECT_ROOT / "pharmacy_forecasting"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Datasets

In [2]:
sales = pd.read_csv(DATA_DIR / "sales_transactions.csv")
stock = pd.read_csv(DATA_DIR / "stock_receipts.csv")
print("Sales shape:", sales.shape)
print("Stock shape:", stock.shape)
sales.head()

Sales shape: (813021, 9)
Stock shape: (3120, 10)


,transaction_id,transaction_date,facility_id,facility_name,drug_id,drug_name,category,quantity_dispensed,unit_price
0,AGHC-20190101-00001,2019-01-01,AGHC-IPD,Allan Galpin Health Center,D001,Artemether-Lumefantrine 20/120mg,Antimalarial,7,71.76
1,AGHC-20190101-00002,2019-01-01,AGHC-MCH,Allan Galpin Health Center,D001,Artemether-Lumefantrine 20/120mg,Antimalarial,5,59.00
2,AGHC-20190101-00003,2019-01-01,AGHC-IPD,Allan Galpin Health Center,D001,Artemether-Lumefantrine 20/120mg,Antimalarial,1,39.49
3,AGHC-20190101-00004,2019-01-01,AGHC-MCH,Allan Galpin Health Center,D001,Artemether-Lumefantrine 20/120mg,Antimalarial,7,4.47
4,AGHC-20190101-00005,2019-01-01,AGHC-OPD,Allan Galpin Health Center,D001,Artemether-Lumefantrine 20/120mg,Antimalarial,7,49.39


## 3. Convert Date Columns and Sort

In [3]:
sales["transaction_date"] = pd.to_datetime(sales["transaction_date"])
stock["stock_received_date"] = pd.to_datetime(stock["stock_received_date"])
sales = sales.sort_values(["drug_id", "transaction_date"]).reset_index(drop=True)
stock = stock.sort_values(["drug_id", "stock_received_date"]).reset_index(drop=True)
sales["transaction_date"].min(), sales["transaction_date"].max()

(Timestamp('2019-01-01 00:00:00'), Timestamp('2024-12-31 00:00:00'))

## 4. Aggregate to Monthly Demand

Demand is **derived** from transaction data: sum of `quantity_dispensed` per drug per month (freq=`"MS"` = month start).

In [4]:
monthly_demand = (
    sales.groupby(["drug_id", pd.Grouper(key="transaction_date", freq="MS")])["quantity_dispensed"]
    .sum()
    .reset_index()
)
monthly_demand = monthly_demand.rename(
    columns={
        "transaction_date": "month",
        "quantity_dispensed": "monthly_demand"
    }
)

monthly_demand.head(15)

,drug_id,month,monthly_demand
0,D001,2019-01-01,2478
1,D001,2019-02-01,2692
2,D001,2019-03-01,3052
3,D001,2019-04-01,2941
4,D001,2019-05-01,3075
5,D001,2019-06-01,2853
6,D001,2019-07-01,2515
7,D001,2019-08-01,2483
8,D001,2019-09-01,2788
9,D001,2019-10-01,3097


## 5. Ensure Continuous Monthly Index per Drug

Fill missing months with 0 (e.g. stockout or no transactions).

In [5]:
def ensure_continuous_months(df: pd.DataFrame, date_col: str = "month", value_col: str = "monthly_demand") -> pd.DataFrame:
    """For each drug_id, create a full range of months and fill missing with 0."""
    all_months = pd.date_range(df[date_col].min(), df[date_col].max(), freq="MS")
    drug_ids = df["drug_id"].unique()
    rows = []
    for did in drug_ids:
        sub = df[df["drug_id"] == did].set_index(date_col).reindex(all_months, fill_value=0)
        sub["drug_id"] = did
        sub = sub.reset_index().rename(columns={"index": date_col})
        sub[value_col] = sub[value_col].fillna(0).astype(int)
        rows.append(sub)
    out = pd.concat(rows, ignore_index=True)
    return out[["drug_id", date_col, value_col]]

In [6]:
monthly_demand = ensure_continuous_months(monthly_demand)
monthly_demand = monthly_demand.sort_values(["drug_id", "month"]).reset_index(drop=True)
monthly_demand.head(20)

,drug_id,month,monthly_demand
0,D001,2019-01-01,2478
1,D001,2019-02-01,2692
2,D001,2019-03-01,3052
3,D001,2019-04-01,2941
4,D001,2019-05-01,3075
5,D001,2019-06-01,2853
6,D001,2019-07-01,2515
7,D001,2019-08-01,2483
8,D001,2019-09-01,2788
9,D001,2019-10-01,3097


## 6. Save Clean Monthly Dataset

In [7]:
out_path = OUTPUTS_DIR / "monthly_demand.csv"
monthly_demand.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print("Shape:", monthly_demand.shape)
print("Columns:", list(monthly_demand.columns))
monthly_demand.tail(10)

Saved: C:\Users\hp\Desktop\prototype2\pharmacy_forecasting\outputs\monthly_demand.csv
Shape: (2520, 3)
Columns: ['drug_id', 'month', 'monthly_demand']


,drug_id,month,monthly_demand
2510,D035,2024-03-01,317
2511,D035,2024-04-01,329
2512,D035,2024-05-01,337
2513,D035,2024-06-01,317
2514,D035,2024-07-01,369
2515,D035,2024-08-01,361
2516,D035,2024-09-01,370
2517,D035,2024-10-01,360
2518,D035,2024-11-01,374
2519,D035,2024-12-01,379


## 7. Inventory balance — opening stock, receipts, and dispensing

In addition to monthly demand, we compute a **per-drug stock balance** using:

\[
\text{current\_stock} = \text{opening\_stock\_units} + \text{total\_received} - \text{total\_dispensed}
\]

Inputs:
- `data/opening_stock.csv`
- `data/stock_receipts.csv`
- `data/sales_transactions.csv`

Output: `outputs/stock_status.csv` for use in the dashboard.

In [8]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")

# Load opening stock, receipts, and sales
opening_stock = pd.read_csv(DATA_DIR / "opening_stock.csv")
stock_raw = pd.read_csv(DATA_DIR / "stock_receipts.csv")
sales_raw = pd.read_csv(DATA_DIR / "sales_transactions.csv")

print("Opening stock shape:", opening_stock.shape)
print("Stock receipts shape:", stock_raw.shape)
print("Sales shape:", sales_raw.shape)
opening_stock.head()

Opening stock shape: (35, 6)
Stock receipts shape: (3120, 10)
Sales shape: (813021, 9)


,drug_id,drug_name,category,opening_stock_units,facility_name,as_of_date
0,D001,Artemether-Lumefantrine 20/120mg,Antimalarial,4420,Allan Galpin Health Center,2019-01-01
1,D002,Artesunate Injection 60mg,Antimalarial,2580,Allan Galpin Health Center,2019-01-01
2,D003,Dihydroartemisinin-Piperaquine,Antimalarial,1900,Allan Galpin Health Center,2019-01-01
3,D004,Quinine Sulphate 300mg,Antimalarial,1764,Allan Galpin Health Center,2019-01-01
4,D005,Sulfadoxine-Pyrimethamine,Antimalarial,3840,Allan Galpin Health Center,2019-01-01


In [9]:
# Standardise dtypes and keys
opening_stock["drug_id"] = opening_stock["drug_id"].astype(str)
opening_stock["opening_stock_units"] = pd.to_numeric(opening_stock["opening_stock_units"], errors="coerce").fillna(0).astype(int)

stock_raw["drug_id"] = stock_raw["drug_id"].astype(str)
stock_raw["quantity_received"] = pd.to_numeric(stock_raw["quantity_received"], errors="coerce").fillna(0).astype(int)

sales_raw["drug_id"] = sales_raw["drug_id"].astype(str)
sales_raw["quantity_dispensed"] = pd.to_numeric(sales_raw["quantity_dispensed"], errors="coerce").fillna(0).astype(int)

In [10]:
# Aggregate totals per drug
total_received = (
    stock_raw.groupby(["drug_id", "drug_name"], as_index=False)["quantity_received"]
    .sum()
    .rename(columns={"quantity_received": "total_received"})
)

total_dispensed = (
    sales_raw.groupby(["drug_id", "drug_name"], as_index=False)["quantity_dispensed"]
    .sum()
    .rename(columns={"quantity_dispensed": "total_dispensed"})
)

# Merge opening stock with totals
stock_status = opening_stock.merge(total_received, on=["drug_id", "drug_name"], how="left")
stock_status = stock_status.merge(total_dispensed, on=["drug_id", "drug_name"], how="left")

stock_status["total_received"] = stock_status["total_received"].fillna(0).astype(int)
stock_status["total_dispensed"] = stock_status["total_dispensed"].fillna(0).astype(int)

stock_status["current_stock"] = (
    stock_status["opening_stock_units"].astype(int)
    + stock_status["total_received"].astype(int)
    - stock_status["total_dispensed"].astype(int)
)
stock_status["current_stock"] = stock_status["current_stock"].clip(lower=0)

stock_status

,drug_id,drug_name,category,opening_stock_units,facility_name,as_of_date,total_received,total_dispensed,current_stock
0,D001,Artemether-Lumefantrine 20/120mg,Antimalarial,4420,Allan Galpin Health Center,2019-01-01,216835,218251,3004
1,D002,Artesunate Injection 60mg,Antimalarial,2580,Allan Galpin Health Center,2019-01-01,99432,100498,1514
2,D003,Dihydroartemisinin-Piperaquine,Antimalarial,1900,Allan Galpin Health Center,2019-01-01,81913,83185,628
3,D004,Quinine Sulphate 300mg,Antimalarial,1764,Allan Galpin Health Center,2019-01-01,56418,52419,5763
4,D005,Sulfadoxine-Pyrimethamine,Antimalarial,3840,Allan Galpin Health Center,2019-01-01,127184,128808,2216
5,D006,Amoxicillin 500mg Capsules,Antibiotic,6720,Allan Galpin Health Center,2019-01-01,252135,213205,45650
6,D007,Azithromycin 250mg Tablets,Antibiotic,2925,Allan Galpin Health Center,2019-01-01,133270,118230,17965
7,D008,Ceftriaxone Injection 1g,Antibiotic,1716,Allan Galpin Health Center,2019-01-01,66703,54923,13496
8,D009,Ciprofloxacin 500mg Tablets,Antibiotic,2660,Allan Galpin Health Center,2019-01-01,103700,87306,19054
9,D010,Metronidazole 400mg Tablets,Antibiotic,4455,Allan Galpin Health Center,2019-01-01,169204,135091,38568


In [11]:
# Basic validation: preview and summary
print("Stock status shape:", stock_status.shape)
print(stock_status.head())
print("\nCurrent stock summary:")
print(stock_status["current_stock"].describe())

Stock status shape: (35, 9)
  drug_id                         drug_name      category  \
0    D001  Artemether-Lumefantrine 20/120mg  Antimalarial   
1    D002         Artesunate Injection 60mg  Antimalarial   
2    D003    Dihydroartemisinin-Piperaquine  Antimalarial   
3    D004            Quinine Sulphate 300mg  Antimalarial   
4    D005         Sulfadoxine-Pyrimethamine  Antimalarial   

   opening_stock_units               facility_name  as_of_date  \
0                 4420  Allan Galpin Health Center  2019-01-01   
1                 2580  Allan Galpin Health Center  2019-01-01   
2                 1900  Allan Galpin Health Center  2019-01-01   
3                 1764  Allan Galpin Health Center  2019-01-01   
4                 3840  Allan Galpin Health Center  2019-01-01   

   total_received  total_dispensed  current_stock  
0          216835           218251           3004  
1           99432           100498           1514  
2           81913            83185            628  


In [12]:
from pathlib import Path

OUTPUTS_DIR = Path("../outputs")

# Save processed stock status for dashboard use
stock_out_path = OUTPUTS_DIR / "stock_status.csv"
stock_status.to_csv(stock_out_path, index=False)
print(f"Saved: {stock_out_path}")
stock_status

Saved: ..\outputs\stock_status.csv


,drug_id,drug_name,category,opening_stock_units,facility_name,as_of_date,total_received,total_dispensed,current_stock
0,D001,Artemether-Lumefantrine 20/120mg,Antimalarial,4420,Allan Galpin Health Center,2019-01-01,216835,218251,3004
1,D002,Artesunate Injection 60mg,Antimalarial,2580,Allan Galpin Health Center,2019-01-01,99432,100498,1514
2,D003,Dihydroartemisinin-Piperaquine,Antimalarial,1900,Allan Galpin Health Center,2019-01-01,81913,83185,628
3,D004,Quinine Sulphate 300mg,Antimalarial,1764,Allan Galpin Health Center,2019-01-01,56418,52419,5763
4,D005,Sulfadoxine-Pyrimethamine,Antimalarial,3840,Allan Galpin Health Center,2019-01-01,127184,128808,2216
5,D006,Amoxicillin 500mg Capsules,Antibiotic,6720,Allan Galpin Health Center,2019-01-01,252135,213205,45650
6,D007,Azithromycin 250mg Tablets,Antibiotic,2925,Allan Galpin Health Center,2019-01-01,133270,118230,17965
7,D008,Ceftriaxone Injection 1g,Antibiotic,1716,Allan Galpin Health Center,2019-01-01,66703,54923,13496
8,D009,Ciprofloxacin 500mg Tablets,Antibiotic,2660,Allan Galpin Health Center,2019-01-01,103700,87306,19054
9,D010,Metronidazole 400mg Tablets,Antibiotic,4455,Allan Galpin Health Center,2019-01-01,169204,135091,38568
